<a href="https://colab.research.google.com/github/heoconngoc/Ruled-Based-A.I.-and-Deep-Learning/blob/main/MINIMAX_TicTacToe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================================================
# TITLE: Minimax Tic Tac Toe
# =========================================================

"""
This notebook implements full Minimax search.
"""

import time
import random

EMPTY=" "
X="X"
O="O"

def create_board():
    return [EMPTY]*9

def get_moves(b):
    return [i for i in range(9) if b[i]==EMPTY]

def check_winner(b):
    wins=[(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]
    for i,j,k in wins:
        if b[i] != EMPTY and b[i] == b[j] == b[k]:
            return b[i]
    return None

def switch(p):
    return O if p==X else X

In [2]:
# ---------------------------
# MINIMAX
# ---------------------------
nodes = 0

def minimax(board, player):
    global nodes
    nodes += 1

    # Terminal state
    winner = check_winner(board)
    if winner == X:
        return 1, None
    if winner == O:
        return -1, None
    if not get_moves(board):
        return 0, None

    # Max branch
    if player == X:
        best_val = -999
        best_move = None
        for m in get_moves(board):
            b = board.copy()
            b[m] = X
            val,_ = minimax(b, O)
            if val > best_val:
                best_val = val
                best_move = m
        return best_val, best_move

    # Min branch
    else:
        best_val = 999
        best_move = None
        for m in get_moves(board):
            b = board.copy()
            b[m] = O
            val,_ = minimax(b, X)
            if val < best_val:
                best_val = val
                best_move = m
        return best_val, best_move


def minimax_agent(board, player):
    _, move = minimax(board, player)
    return move

In [4]:
# ---------------------------
# GAME ENGINE
# ---------------------------
def play_game(agent1, agent2, verbose=False):
    board = create_board()
    player = X

    while True:

        if player == X:
            move = agent1(board, X)
        else:
            move = agent2(board, O)

        board[move] = player

        w = check_winner(board)
        if w:
            return 1 if w == X else -1

        if not get_moves(board):
            return 0

        player = switch(player)

In [5]:
def evaluate(agent1, agent2, games=100):

    results = []
    total_nodes_before = nodes

    start_time = time.time()

    for i in range(games):

        # swap first move for fairness
        if i % 2 == 0:
            res = play_game(agent1, agent2)
        else:
            res = play_game(agent2, agent1)
            res = -res

        results.append(res)

    end_time = time.time()

    total_nodes_after = nodes

    print("\n======================")
    print("RESULTS")
    print("======================")

    print("Agent1 Wins :", results.count(1))
    print("Agent2 Wins :", results.count(-1))
    print("Ties        :", results.count(0))

    print("\nTime taken  :", round(end_time - start_time, 4), "sec")
    print("Avg time/game:", round((end_time - start_time)/games, 6), "sec")

    print("\nNodes explored:", total_nodes_after - total_nodes_before)
    print("Avg nodes/game:", (total_nodes_after - total_nodes_before)/games)

In [6]:
# Random agent

def random_agent(board, player):
    return random.choice(get_moves(board))

# Rule-based 1-step agent

def rule_1_step(board, player):

    opponent = switch(player)

    # win
    for m in get_moves(board):
        b = board.copy()
        b[m] = player
        if check_winner(b) == player:
            return m

    # block
    for m in get_moves(board):
        b = board.copy()
        b[m] = opponent
        if check_winner(b) == opponent:
            return m

    return random.choice(get_moves(board))

In [7]:
# Minimax vs random
nodes = 0
evaluate(minimax_agent, random_agent, games=100)


RESULTS
Agent1 Wins : 86
Agent2 Wins : 0
Ties        : 14

Time taken  : 60.9789 sec
Avg time/game: 0.609789 sec

Nodes explored: 30975432
Avg nodes/game: 309754.32


In [8]:
# Minimax vs 1-step AI
nodes = 0
evaluate(minimax_agent, rule_1_step, games=100)


RESULTS
Agent1 Wins : 56
Agent2 Wins : 0
Ties        : 44

Time taken  : 56.7552 sec
Avg time/game: 0.567552 sec

Nodes explored: 31002201
Avg nodes/game: 310022.01


In [9]:
# Minimax vs Minimax
nodes = 0
evaluate(minimax_agent, minimax_agent, games=100)


RESULTS
Agent1 Wins : 0
Agent2 Wins : 0
Ties        : 100

Time taken  : 114.2139 sec
Avg time/game: 1.142139 sec

Nodes explored: 61818400
Avg nodes/game: 618184.0


Minimax guarantees optimal play but explores the full game tree, resulting in high computational cost even for small games.